In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from scipy import ndimage

In [ ]:
CAMINHO_DATASET = "../data/processed/dataset_segmentacao.csv"

PASTA_IMAGENS = "../data/segmentacao/images"
PASTA_MASCARAS = "../data/segmentacao/masks"

dataset = pd.read_csv(CAMINHO_DATASET)

positivas = dataset[
    dataset["possui_banco"] == True
].copy()

print(f"Imagens positivas: {len(positivas)}")

In [ ]:
def calcular_indice(a, b):
    a = a.astype(np.float32)
    b = b.astype(np.float32)

    denominador = a + b

    return np.divide(
        a - b,
        denominador,
        out=np.zeros_like(a, dtype=np.float32),
        where=np.abs(denominador) > 1e-6
    )

In [ ]:
def gerar_mascara_inicial(caminho_imagem):

    with rasterio.open(caminho_imagem) as src:
        b02 = src.read(1).astype(np.float32)
        b03 = src.read(2).astype(np.float32)
        b04 = src.read(3).astype(np.float32)
        b08 = src.read(4).astype(np.float32)
        b11 = src.read(5).astype(np.float32)
        b12 = src.read(6).astype(np.float32)

    ndvi = calcular_indice(
        b08,
        b04
    )

    ndwi = calcular_indice(
        b03,
        b08
    )

    mndwi = calcular_indice(
        b03,
        b11
    )

    numerador_bsi = (
        (b11 + b04)
        - (b08 + b02)
    )

    denominador_bsi = (
        (b11 + b04)
        + (b08 + b02)
    )

    bsi = np.divide(
        numerador_bsi,
        denominador_bsi,
        out=np.zeros_like(
            b04,
            dtype=np.float32
        ),
        where=np.abs(
            denominador_bsi
        ) > 1e-6
    )

    # Brilho médio no visível
    brilho = (
        b02 + b03 + b04
    ) / 3

    # Considera como claro apenas pixels
    # acima do percentil 70 da própria imagem
    limite_brilho = np.nanpercentile(
        brilho,
        70
    )

    mascara = (
    (ndvi < 0.30)
    & (ndwi < 0.15)
    & (mndwi < 0.15)
    & (bsi > -0.10)
    & (brilho >= limite_brilho)
)

mascara = ndimage.binary_closing(
    mascara,
    structure=np.ones((3, 3))
)

mascara = ndimage.binary_fill_holes(
    mascara
)

rotulos, numero = ndimage.label(
    mascara
)

if numero > 0:

    tamanhos = ndimage.sum(
        mascara,
        rotulos,
        index=np.arange(1, numero + 1)
    )

    mascara_limpa = np.zeros_like(
        mascara,
        dtype=bool
    )

    for rotulo, tamanho in enumerate(
        tamanhos,
        start=1
    ):

        if tamanho >= 20:
            mascara_limpa[
                rotulos == rotulo
            ] = True

    mascara = mascara_limpa

return mascara.astype(np.uint8)

In [ ]:
def normalizar(banda):
    minimo = np.nanpercentile(
        banda,
        2
    )

    maximo = np.nanpercentile(
        banda,
        98
    )

    banda = np.clip(
        banda,
        minimo,
        maximo
    )

    return (
        banda - minimo
    ) / (
        maximo - minimo + 1e-8
    )

In [ ]:
def visualizar_mascara(
    caminho_imagem,
    mascara,
    titulo=""
):

    with rasterio.open(
        caminho_imagem
    ) as src:

        b02 = src.read(1)
        b03 = src.read(2)
        b04 = src.read(3)

    rgb = np.dstack([
        normalizar(b04),
        normalizar(b03),
        normalizar(b02)
    ])

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 6)
    )

    axes[0].imshow(rgb)
    axes[0].set_title(
        "RGB"
    )

    axes[1].imshow(
        mascara,
        cmap="gray"
    )
    axes[1].set_title(
        "Máscara inicial"
    )

    axes[2].imshow(rgb)
    axes[2].imshow(
        mascara,
        alpha=0.35,
        cmap="Reds"
    )
    axes[2].set_title(
        "Sobreposição"
    )

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        titulo
    )

    plt.tight_layout()
    plt.show()

In [ ]:
id_imagem = str(
    positivas.iloc[0]["id"]
).replace(".0", "")

caminho = os.path.join(
    PASTA_IMAGENS,
    f"{id_imagem}.tif"
)

mascara = gerar_mascara_inicial(
    caminho
)

visualizar_mascara(
    caminho,
    mascara,
    titulo=f"Imagem {id_imagem}"
)